# Holdout Evaluation

Evaluate the selected edit on holdout prompts and inspect topic-level failure patterns and sample responses.

**Runtime:** T4 GPU

In [ ]:
import os
import subprocess
import sys
from getpass import getpass
from pathlib import Path

# --- Kaggle environment ---
try:
    from kaggle_secrets import UserSecretsClient
    kaggle_secrets = UserSecretsClient()
except ImportError:
    kaggle_secrets = None

try:
    from google.colab import userdata as colab_userdata
except ImportError:
    colab_userdata = None


def read_secret(name: str) -> str:
    if kaggle_secrets is not None:
        try:
            return kaggle_secrets.get_secret(name).strip()
        except Exception:
            pass
    if colab_userdata is not None:
        try:
            return colab_userdata.get(name).strip()
        except Exception:
            pass
    return os.environ.get(name, "").strip()


REPO_URL = "https://github.com/aaliyan1230/false-refusal-suppression.git"

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN:
    try:
        HF_TOKEN = read_secret("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass("Enter your HF token (or press Enter to skip): ")

# Detect Kaggle vs Colab
REPO_DIR = Path("/kaggle/working/false-refusal-suppression") if Path("/kaggle").exists() else Path("/content/false-refusal-suppression")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "HEAD"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[train,dev]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "transformers>=4.52", "accelerate"], check=True)

os.chdir(REPO_DIR)
src_path = REPO_DIR / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import transformers
print(f"transformers version: {transformers.__version__}")
print(REPO_DIR)
print("HF token loaded:", bool(HF_TOKEN))

In [ ]:
from pathlib import Path

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
SPLIT_DIR = Path("data/processed/splits/orbench")
DIRECTION_ARTIFACT = Path("artifacts/directions/llama31_8b_bf16_orbench_unsafe_vs_easy.json")
SEARCH_ARTIFACT = Path("artifacts/edits/llama31_8b_bf16_orbench_search.json")
EVAL_ARTIFACT = Path("artifacts/eval/llama31_8b_bf16_orbench_holdout.json")

required_paths = [SPLIT_DIR / "holdout.jsonl", DIRECTION_ARTIFACT, SEARCH_ARTIFACT]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    print(f"⚠ Missing required inputs: {missing}")
    print("Run notebook 20 first to generate these artifacts.")
else:
    print("All required inputs found ✓")

In [ ]:
import subprocess, sys

# Holdout eval: 2 passes (base + best edit) on up to 200 prompts
# T4 with 4-bit: ~200 prompts × 2 passes ≈ 30-40 minutes
cmd = [
    sys.executable,
    "scripts/run_eval.py",
    "--model-id",       MODEL_ID,
    "--prompts",        str(SPLIT_DIR / "holdout.jsonl"),
    "--direction-artifact", str(DIRECTION_ARTIFACT),
    "--candidate-json", str(SEARCH_ARTIFACT),
    "--output",         str(EVAL_ARTIFACT),
    "--load-in-4bit",
    "--prompt-limit",   "200",
]
print("Running:", " ".join(cmd), flush=True)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)

rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"run_eval.py failed with exit code {rc}")
print("\n✓ Holdout evaluation completed")

In [ ]:
import json
import pandas as pd

with open(EVAL_ARTIFACT, "r", encoding="utf-8") as handle:
    report = json.load(handle)

print("=== Overall Metrics ===")
display(pd.DataFrame([report["metrics"]]))

print("\n=== Group-Level Metrics ===")
if "group_metrics" in report:
    display(pd.DataFrame([report["group_metrics"]]))

print("\n=== Topic Breakdown (sorted by refusal rate) ===")
topic_df = pd.DataFrame.from_dict(report["topic_breakdown"], orient="index")
if "refusal_rate" in topic_df.columns:
    topic_df = topic_df.sort_values("refusal_rate", ascending=False)
display(topic_df)

print("\n=== Sample Responses ===")
samples_df = pd.DataFrame(report["response_samples"])
display(samples_df)

In [ ]:
# Show the search candidates from nb20 for context
import json
import pandas as pd

with open(SEARCH_ARTIFACT, "r", encoding="utf-8") as f:
    candidates = json.load(f)

print(f"=== Edit Search Results ({len(candidates)} candidates) ===")
cdf = pd.DataFrame(candidates)[["name", "false_refusal_rate", "true_refusal_rate", "capability_retention", "harmless_kl_penalty"]]
display(cdf)

print(f"\nBest candidate (used for holdout eval): {candidates[0]['name']}")
print(f"  false_refusal_rate: {candidates[0]['false_refusal_rate']:.3f}")
print(f"  true_refusal_rate:  {candidates[0]['true_refusal_rate']:.3f}")
print(f"  capability_retention: {candidates[0]['capability_retention']:.3f}")